# Tuning NeuralProphet

We saw in 2neural_solo_prophet.ipynb that NeuralProphet outperformed Prophet. The margin was not very wide, but we might be able to do much better if we tune the parameters of NeuralProphet a lot more. This is the purpose of this notebook.

# Import Packages

In [1]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [2]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [3]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [4]:
params = {'changepoint_prior_scale': 0.5, 'seasonality_prior_scale': 5, 'holidays': holidays}

In [5]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(**params)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_pred = np.round(y_pred)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet_results_df = pd.DataFrame(results)
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

15:00:55 - cmdstanpy - INFO - Chain [1] start processing
15:00:55 - cmdstanpy - INFO - Chain [1] done processing
15:00:55 - cmdstanpy - INFO - Chain [1] start processing
15:00:56 - cmdstanpy - INFO - Chain [1] done processing
15:00:56 - cmdstanpy - INFO - Chain [1] start processing
15:00:57 - cmdstanpy - INFO - Chain [1] done processing
15:00:57 - cmdstanpy - INFO - Chain [1] start processing
15:00:58 - cmdstanpy - INFO - Chain [1] done processing
15:00:58 - cmdstanpy - INFO - Chain [1] start processing
15:00:58 - cmdstanpy - INFO - Chain [1] done processing
15:00:59 - cmdstanpy - INFO - Chain [1] start processing
15:00:59 - cmdstanpy - INFO - Chain [1] done processing
15:01:00 - cmdstanpy - INFO - Chain [1] start processing
15:01:00 - cmdstanpy - INFO - Chain [1] done processing
15:01:01 - cmdstanpy - INFO - Chain [1] start processing
15:01:01 - cmdstanpy - INFO - Chain [1] done processing
15:01:02 - cmdstanpy - INFO - Chain [1] start processing
15:01:02 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,10.285218,0.135271
1,1,15.675276,0.269553
2,2,12.850792,0.203628
3,3,13.851251,0.138456
4,4,11.991068,0.103076
5,5,16.550788,0.157929
6,6,14.665314,0.165510
7,7,13.799586,0.149353
8,8,16.743869,0.233097
9,9,11.398621,0.130736


# Optuna Hyperparamter Tuning for Neural Prophet

**final_model_neural.db** Optuna is able to save its previous trials and study to a file and reload its studies one future runs. This information is held inside of "final_model_neural.db". Optuna was designed with flexibility in mind e.g. adding new parameters to teach trial, changing the suggested values for each trial, and changing the sizes of the time series split will not break anything.

In [6]:
# Open rat sightings data
rat_sighting = pd.read_csv("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")


# Clean the data
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

# Download the weather data

lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)


# Add the weather data to rs

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])

rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")



# Time Series Split. Set to n_splits = 2 to reduce run time. Make higher to avoid overfitting.
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

WARNING - (py.warnings._showwarnmsg) - C:\Users\daoke\AppData\Local\Temp\ipykernel_17564\1432134030.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date'])



In [7]:
def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 365),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 365),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 365),
    }
    n_lags = trial.suggest_int('n_lags', 14, 365)
    epochs = trial.suggest_int('epochs', 50, 200)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.5, log=True)
    batch_size = trial.suggest_int('batch_size', 128, 1024)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 10)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        # intermediate_score = np.mean(fold_rmses)
        # trial.report(intermediate_score, i)
        # if trial.should_prune():
        #     raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(
    direction="minimize",
    study_name="final_model_neural",
    storage="sqlite:///final_model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=1)  # adjust n_trials as needed

best_params = study.best_params

print(best_params)

[I 2026-03-14 15:01:34,555] A new study created in RDB with name: final_model_neural
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.934% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


...

Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
[I 2026-03-14 15:05:57,511] Trial 0 finished with value: 20.143677791763327 and parameters: {'lag_temp_max': 93, 'lag_temp_min': 287, 'lag_snowfall': 316, 'n_lags': 356, 'epochs': 136, 'learning_rate': 0.044785265087907485, 'ar_reg': 9.268440303714137}. Best is trial 0 with value: 20.143677791763327.


{'lag_temp_max': 93, 'lag_temp_min': 287, 'lag_snowfall': 316, 'n_lags': 356, 'epochs': 136, 'learning_rate': 0.044785265087907485, 'ar_reg': 9.268440303714137}


# Check Performance

In [11]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()


if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [12]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.934% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.935% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


...

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.utils.configure_trainer) - Using accelerator gpu with -1 device(s).


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 2it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


# Results

In [16]:
results

[{'fold': 0, 'rmse': 18.838583875430782, 'mape': 0.26427388191223145},
 {'fold': 1, 'rmse': 19.852308839522056, 'mape': 0.3265976905822754},
 {'fold': 2, 'rmse': 24.97110635001481, 'mape': 0.3705580234527588},
 {'fold': 3, 'rmse': 27.054634484138063, 'mape': 0.3067665100097656},
 {'fold': 4, 'rmse': 17.54580448788123, 'mape': 0.190115824341774},
 {'fold': 5, 'rmse': 20.63877824582814, 'mape': 0.1564193069934845},
 {'fold': 6, 'rmse': 19.236154308186432, 'mape': 0.19858188927173615},
 {'fold': 7, 'rmse': 15.606427438481177, 'mape': 0.1417388767004013},
 {'fold': 8, 'rmse': 16.319797330607674, 'mape': 0.1984344720840454},
 {'fold': 9, 'rmse': 39.85592185404745, 'mape': 0.41948410868644714},
 {'fold': 10, 'rmse': 24.199010352714453, 'mape': 0.21518735587596893},
 {'fold': 11, 'rmse': 20.535653720553157, 'mape': 0.21237008273601532},
 {'fold': 12, 'rmse': 25.79498687063535, 'mape': 0.28722208738327026},
 {'fold': 13, 'rmse': 23.833586720362323, 'mape': 0.22995033860206604},
 {'fold': 14, '

In [17]:
df = pd.DataFrame(results)[['fold', 'rmse']]

# add mean rmse row
df.loc[len(df)] = ['mean', df['rmse'].mean()]

df

,fold,rmse
0,0,18.838584
1,1,19.852309
2,2,24.971106
3,3,27.054634
4,4,17.545804
5,5,20.638778
6,6,19.236154
7,7,15.606427
8,8,16.319797
9,9,39.855922
